Run script
not debugged yet

In [ ]:
import os
import pickle
import numpy as np
import scipy.sparse as sp
from datetime import date
from multiprocessing import Pool

# Import user-defined functions (assumed available)
from blinking_utils import Determine_Blinking_Distribution5, DDC_MCMC

# Configuration
cluster = False  # Set True if using a cluster environment
Resolution = 40             # Determined via New_Determine_Res
stepper = 100               # Maximum number of MCMC steps
N_f = 100                   # Determined via Determine_N
Photon_weighted_Correction = True

# Load data: LocalizationsFinal, Frame_Information, filename
# (User should define how to load these, e.g., from .mat or .npz files)
# Example:
# data = np.load('data.npz', allow_pickle=True)
# LocalizationsFinal = data['LocalizationsFinal']
# Frame_Information = data['Frame_Information']
# filename = data['filename'].item()

# Placeholder for loaded data
LocalizationsFinal = []
Frame_Information = []
filename = 'experiment'
TrueLocalizations = []

# Initialize arrays
addonarray = np.zeros(500, dtype=int)
Photons = [[] for _ in Frame_Information]

# If Photons was not provided, initialize with ones
if not any(Photons):
    Photons = [np.ones(len(frames), dtype=float) for frames in Frame_Information]

# Initialize TrueLocalizations if empty
if not TrueLocalizations:
    TrueLocalizations = [[] for _ in Frame_Information]

# Ensure 3D localizations
for idx, loc in enumerate(LocalizationsFinal):
    loc = np.asarray(loc)
    if loc.shape[1] < 3:
        zcol = np.zeros((loc.shape[0], 1))
        LocalizationsFinal[idx] = np.hstack([loc, zcol])

# Prepare output containers
Final_Loc_Blinking_Corrected = [None] * len(Frame_Information)
Final_Frame_Blinking_Corrected = [None] * len(Frame_Information)
Trajectory_of_Localizations = [None] * len(Frame_Information)

# Determine blinking distribution
bins, Distribution_for_Blink, _, Resolution, X_overall, M_mat = \
    Determine_Blinking_Distribution5(
        LocalizationsFinal,
        Frame_Information,
        N_f,
        Resolution
    )

# Timer for each image (seconds)
timer = 3600 * 0.2

# Initialize MCMC tracking structures
n_images = len(Frame_Information)
Constantf = [None] * n_images
Constantf2 = [None] * n_images
Orderf = [None] * n_images
Step = np.zeros(n_images, dtype=int)
Lik = [None] * n_images
RelScore = [None] * n_images
Numb_of_Loc = [None] * n_images
Prob_dists = [dict(Deviation_in_Probability=None,
                   Prob_Distributions=None,
                   Dscale_store=None)
              for _ in range(n_images)]

# Prepare save path
today_str = date.today().isoformat()
output_name = f"Analyzed_Time_{today_str}_{filename}.pkl"

# Save initial state
with open(output_name, 'wb') as f:
    pickle.dump({
        'Constantf': Constantf,
        'Constantf2': Constantf2,
        'Orderf': Orderf,
        'Step': Step,
        'Lik': Lik,
        'RelScore': RelScore,
        'Numb_of_Loc': Numb_of_Loc,
        'Prob_dists': Prob_dists,
        'Final_Loc_Blinking_Corrected': Final_Loc_Blinking_Corrected,
        'Final_Frame_Blinking_Corrected': Final_Frame_Blinking_Corrected,
        'Trajectory_of_Localizations': Trajectory_of_Localizations,
    }, f)

# Blinking elimination loop
def process_image(args):
    ksu = args
    if len(LocalizationsFinal[ksu]) <= 10 or Step[ksu] >= stepper:
        Step[ksu] = stepper
        return None

    locs = LocalizationsFinal[ksu]
    frames = np.round(Frame_Information[ksu]).astype(int)
    if len(locs) > 8000:
        print(f"Warning: too many localizations in image {ksu}; consider splitting")

    result = DDC_MCMC(
        index=ksu,
        locs=locs,
        frames=frames,
        N_f=N_f,
        Resolution=Resolution,
        TrueLoc=TrueLocalizations[ksu],
        bins=bins,
        blink_dist=Distribution_for_Blink,
        step0=Step[ksu],
        timer=timer,
        Constf0=Constantf[ksu],
        Order0=Orderf[ksu],
        Lik0=Lik[ksu],
        RelScore0=RelScore[ksu],
        Numb0=Numb_of_Loc[ksu],
        Prob0=Prob_dists[ksu],
        stepper=stepper,
        addon=addonarray[ksu],
        Constf2_0=Constantf2[ksu],
        X_overall=X_overall,
        M_mat=M_mat
    )

    (Traj2,
     Final_loc2,
     Final_frame2,
     LikHood,
     Score,
     Numb,
     bestlik,
     steps,
     Constant,
     Order,
     Prob2,
     Constant2) = result

    maxtemp = np.max(Lik[ksu]) if Lik[ksu] is not None else -np.inf

    # Update probability distributions
    Prob_dists[ksu] = {
        'Deviation_in_Probability': sp.csr_matrix(Prob2['Deviation_in_Probability']),
        'Prob_Distributions': sp.csr_matrix(Prob2['Prob_Distributions']),
        'Dscale_store': Prob2['Dscale_store']
    }

    # Store MCMC state
    Constantf[ksu] = Constant
    Constantf2[ksu] = Constant2
    Orderf[ksu] = Order
    Step[ksu] = steps
    Lik[ksu] = LikHood
    RelScore[ksu] = Score
    Numb_of_Loc[ksu] = Numb

    if bestlik > maxtemp:
        if Photon_weighted_Correction:
            new_locs = []
            new_frames = []
            for traj_id in np.unique(Traj2):
                mask = (Traj2 == traj_id)
                weights = Photons[ksu][mask]
                weights = weights / weights.sum()
                wloc = np.average(locs[mask], axis=0, weights=weights)
                new_locs.append(wloc)
                new_frames.append(np.mean(Frame_Information[ksu][mask]))
            Final_loc2 = np.vstack(new_locs)
            Final_frame2 = np.array(new_frames)

        Final_Loc_Blinking_Corrected[ksu] = Final_loc2
        Final_Frame_Blinking_Corrected[ksu] = Final_frame2
        Trajectory_of_Localizations[ksu] = Traj2

    return None

# Main parallel or sequential loop
while np.any(Step < stepper):
    if cluster:
        # Example: use MPI or Slurm array jobs
        pass
    else:
        with Pool() as pool:
            pool.map(process_image, list(range(n_images)))

    # Save intermediate results
    with open(output_name, 'wb') as f:
        pickle.dump({
            'Constantf': Constantf,
            'Constantf2': Constantf2,
            'Orderf': Orderf,
            'Step': Step,
            'Lik': Lik,
            'RelScore': RelScore,
            'Numb_of_Loc': Numb_of_Loc,
            'Prob_dists': Prob_dists,
            'Final_Loc_Blinking_Corrected': Final_Loc_Blinking_Corrected,
            'Final_Frame_Blinking_Corrected': Final_Frame_Blinking_Corrected,
            'Trajectory_of_Localizations': Trajectory_of_Localizations,
        }, f)

# End of script
